In [2]:
import numpy as np
import xarray as xr
import pandas as pd
import os

out = "../Data/Raw/ERA5/era5_geopotential_alps.nc"   # adjust if it was zipped/extracted
zg = xr.open_dataset(out)
z_era5_m = zg["z"].squeeze() / 9.80665                # geopotential -> metres

# ---- reload your Stage 1 table ----
df = pd.read_csv("../Data/Processed/stage1_hugonnet_rgi6_merged.csv")
print("Loaded", len(df), "glaciers")
print("Columns:", df.columns.tolist())

Loaded 3927 glaciers
Columns: ['rgiid', 'period', 'area', 'dmdtda', 'err_dmdtda', 'reg', 'is_cor', 'RGIId', 'Area', 'Zmin', 'Zmax', 'Zmed', 'Slope', 'Aspect', 'Lmax', 'CenLon', 'CenLat', 'northness', 'eastness']


In [4]:
import xarray as xr
import pandas as pd
import numpy as np
import os
lon = xr.DataArray(df["CenLon"].values, dims="g")
lat = xr.DataArray(df["CenLat"].values, dims="g")

# ERA5 grid elevation at each glacier
df["z_era5"] = z_era5_m.sel(longitude=lon, latitude=lat, method="nearest").values

# lapse-rate corrected summer temperature
df["t_jja_corr"] = df["t_jja"] - 0.0065 * (df["Zmed"] - df["z_era5"])

print(df[["Zmed", "z_era5", "t_jja", "t_jja_corr"]].describe())

df.to_csv("../Data/Processed/stage2_model_ready.csv", index=False)
print("\nSaved model-ready table:", len(df), "glaciers")

              Zmed       z_era5        t_jja   t_jja_corr
count  3927.000000  3927.000000  3927.000000  3927.000000
mean   2959.442322  2008.671509    10.789203     4.609192
std     296.430247   309.207672     2.007846     1.823914
min    1716.000000     2.812833     8.086038    -4.701589
25%    2773.000000  1831.887695     9.570605     3.605454
50%    2949.000000  2084.895508    10.573565     4.651069
75%    3132.000000  2194.992188    11.506474     5.643891
max    4451.000000  2425.701416    24.193636    14.337225

Saved model-ready table: 3927 glaciers


In [3]:
import xarray as xr
import pandas as pd
import numpy as np
import os

ERA5_DIR = r"C:\DATA\Dissertation\Glacier_Mass\Glacier_Mass_Balance\Glacier_Mass_Balance\Data\Raw\ERA5\extracted"

# ---- 1. Load the two monthly files ----
ds_t = xr.open_dataset(os.path.join(ERA5_DIR, "data_stream-moda_stepType-avgua.nc"))
ds_a = xr.open_dataset(os.path.join(ERA5_DIR, "data_stream-moda_stepType-avgad.nc"))

# ---- 2. Fix the timestamp misalignment ----
for d in (ds_t, ds_a):
    d["valid_time"] = d["valid_time"].values.astype("datetime64[M]").astype("datetime64[ns]")

# ---- 3. Unit conversions ----
t2m_c = ds_t["t2m"] - 273.15                       # Kelvin -> Celsius
days  = ds_a["valid_time"].dt.days_in_month
tp_mm = ds_a["tp"] * 1000 * days                   # m/day -> mm per month
ssrd  = ds_a["ssrd"]

# ---- 4. Restricting to the Hugonnet window (2000-2019) ----
per = slice("2000-01-01", "2019-12-31")
t2m_c, tp_mm, ssrd = t2m_c.sel(valid_time=per), tp_mm.sel(valid_time=per), ssrd.sel(valid_time=per)

# ---- 5. Climate summaries on the grid ----
jja      = t2m_c["valid_time"].dt.month.isin([6, 7, 8])
t_jja    = t2m_c.sel(valid_time=jja).mean("valid_time")
p_annual = tp_mm.groupby("valid_time.year").sum().mean("year")
srad_jja = ssrd.sel(valid_time=ssrd["valid_time"].dt.month.isin([6, 7, 8])).mean("valid_time")

# ---- 6. Geopotential -> ERA5 cell elevation ----
zg = xr.open_dataset("../Data/Raw/ERA5/era5_geopotential_alps.nc")
z_era5_m = zg["z"].squeeze() / 9.80665

# ---- 7. Extract everything at the 3,927 glacier locations ----
df = pd.read_csv("../Data/Processed/stage1_hugonnet_rgi6_merged.csv")

lon = xr.DataArray(df["CenLon"].values, dims="g")
lat = xr.DataArray(df["CenLat"].values, dims="g")

df["t_jja"]    = t_jja.sel(longitude=lon, latitude=lat, method="nearest").values
df["p_annual"] = p_annual.sel(longitude=lon, latitude=lat, method="nearest").values
df["srad_jja"] = srad_jja.sel(longitude=lon, latitude=lat, method="nearest").values
df["z_era5"]   = z_era5_m.sel(longitude=lon, latitude=lat, method="nearest").values

# ---- 8. Lapse-rate correction ----
df["t_jja_corr"] = df["t_jja"] - 0.0065 * (df["Zmed"] - df["z_era5"])

# ---- 9. Checks ----
print(df[["Zmed", "z_era5", "t_jja", "t_jja_corr", "p_annual"]].describe())
print("\nMissing values:\n", df[["t_jja", "t_jja_corr", "p_annual", "srad_jja"]].isna().sum())

# ---- 10. Save ----
df.to_csv("../Data/Processed/stage2_model_ready.csv", index=False)
print("\nSaved:", len(df), "glaciers with climate features")

              Zmed       z_era5        t_jja   t_jja_corr     p_annual
count  3927.000000  3927.000000  3927.000000  3927.000000  3927.000000
mean   2959.442322  2008.671509    10.789203     4.609192  1567.703170
std     296.430247   309.207672     2.007846     1.823914   288.123875
min    1716.000000     2.812833     8.086038    -4.701589   710.484362
25%    2773.000000  1831.887695     9.570605     3.605454  1366.350174
50%    2949.000000  2084.895508    10.573565     4.651069  1543.352032
75%    3132.000000  2194.992188    11.506474     5.643891  1821.775389
max    4451.000000  2425.701416    24.193636    14.337225  2253.518343

Missing values:
 t_jja         0
t_jja_corr    0
p_annual      0
srad_jja      0
dtype: int64

Saved: 3927 glaciers with climate features
